In [54]:
library("tidyverse")
library("data.table")


Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last


The following object is masked from ‘package:purrr’:

    transpose




In [2]:
dep <- read.table("/u/project/geschwind/davidgib/ace/ptdt/SparkAce_eur_structure.txt", header = T)

In [3]:
head(dep)

,FID,PROBAND_ID,FATHER_ID,MOTHER_ID,UNAFFECTED_ID
,<chr>,<chr>,<chr>,<chr>,<chr>
1,AU0226,AU0226302_AU0226302,NA,NA,AU0226303_AU0226303
2,AU0253,AU025304_AU025304,NA,NA,NA
3,AU0329,AU032904_AU032904,NA,NA,NA
4,AU0347,AU0347301_AU0347301,NA,NA,NA
5,AU0347,AU0347302_AU0347302,NA,NA,NA
6,AU0347,AU0347303_AU0347303,NA,NA,NA


In [4]:
colnames(dep)

[1] "FID"           "PROBAND_ID"    "FATHER_ID"     "MOTHER_ID"    
[5] "UNAFFECTED_ID"

In [5]:
table(table(dep$PROBAND_ID))


    1     2     3     4     5 
33164   245    23    32     3 

In [6]:
table(dep$PROBAND_ID)[table(dep$PROBAND_ID) == 5]


SP0094948 SP0094951 SP0094953 
        5         5         5 

In [7]:
dep$PROBAND_ID == "SP0094948"

[1] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
   [13] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
   [25] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
   [37] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
   [49] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
   [61] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
   [73] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
   [85] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
   [97] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [109] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [121] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [133] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [145] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [157] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [169] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [181] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [193] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [205] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [217] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [229] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [241] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [253] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [265] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [277] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [289] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [301] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [313] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [325] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [337] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [349] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [361] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [373] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [385] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [397] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [409] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [421] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [433] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [445] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [457] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [469] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [481] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [493] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [505] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [517] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [529] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [541] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [553] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [565] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [577] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [589] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [6

In [8]:
dep[dep$PROBAND_ID %in% c("SP0094948"),]

,FID,PROBAND_ID,FATHER_ID,MOTHER_ID,UNAFFECTED_ID
,<chr>,<chr>,<chr>,<chr>,<chr>
10070,SF0094948,SP0094948,SP0094944,SP0095065,SP0094965
10073,SF0094948,SP0094948,SP0094944,SP0095065,SP0094966
10076,SF0094948,SP0094948,SP0094944,SP0095065,SP0094967
10079,SF0094948,SP0094948,SP0094944,SP0095065,SP0094968
10082,SF0094948,SP0094948,SP0094944,SP0095065,SP0094969


In [9]:
dep[dep$PROBAND_ID %in% c("SP0094965"),]

FID,PROBAND_ID,FATHER_ID,MOTHER_ID,UNAFFECTED_ID
<chr>,<chr>,<chr>,<chr>,<chr>


In [10]:
# Use pivot_longer to reshape the data
dep_long <- dep %>%
  pivot_longer(cols = c(PROBAND_ID, FATHER_ID, MOTHER_ID, UNAFFECTED_ID), 
               names_to = "Relation", 
               values_to = "ID") %>%
  filter(!is.na(ID))  # Filter out rows where IDs are missing, if needed


In [11]:
head(dep_long)

FID,Relation,ID
<chr>,<chr>,<chr>
AU0226,PROBAND_ID,AU0226302_AU0226302
AU0226,UNAFFECTED_ID,AU0226303_AU0226303
AU0253,PROBAND_ID,AU025304_AU025304
AU0329,PROBAND_ID,AU032904_AU032904
AU0347,PROBAND_ID,AU0347301_AU0347301
AU0347,PROBAND_ID,AU0347302_AU0347302


In [12]:
table(dep_long$Relation)


    FATHER_ID     MOTHER_ID    PROBAND_ID UNAFFECTED_ID 
        30467         19618         33866          5436 

In [13]:
dep_long <- dep_long %>%
  mutate(relationship = case_when(
    Relation == "FATHER_ID" ~ "father",
    Relation == "MOTHER_ID" ~ "mother",
    Relation == "PROBAND_ID" ~ "sibling",
    Relation == "UNAFFECTED_ID" ~ "sibling"
  ))

In [14]:
head(dep_long)

FID,Relation,ID,relationship
<chr>,<chr>,<chr>,<chr>
AU0226,PROBAND_ID,AU0226302_AU0226302,sibling
AU0226,UNAFFECTED_ID,AU0226303_AU0226303,sibling
AU0253,PROBAND_ID,AU025304_AU025304,sibling
AU0329,PROBAND_ID,AU032904_AU032904,sibling
AU0347,PROBAND_ID,AU0347301_AU0347301,sibling
AU0347,PROBAND_ID,AU0347302_AU0347302,sibling


In [15]:
table(dep_long$relationship)


 father  mother sibling 
  30467   19618   39302 

In [16]:
table(table(dep_long$ID))


    1     2     3     4     5     6     7     8    10    12    15 
75596  5370   546   252    14    33     1     4     2     4     2 

In [43]:
dep_long$IID <- sub(".*_", "", dep_long$ID)

In [18]:
#"/u/project/geschwind/davidgib/ace/ptdt/SparkAce_afr_structure.txt"

In [19]:
aap <- read.csv("20230809_imputation_individuals_jkl.csv")

In [20]:
head(aap)

,FID,IID,file_name,aff,relationship,race,eth,grant,DNA.source,Notes
,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,AU3494,AU3494301,2011-158.clean536p.fam,0,sibling,black-or-african-american,not-hispanic-or-latino,ACE1,LCL,
2,AU3302,AU3302201,2011-158.clean536p.fam,0,father,black-or-african-american,not-hispanic-or-latino,AGRE,cpDNA,
3,AU3302,AU3302302,2011-158.clean536p.fam,2,sibling,black-or-african-american,not-hispanic-or-latino,AGRE,cpDNA,
4,AU3570,AU3570302,2011-158.clean536p.fam,2,sibling,black-or-african-american,not-hispanic-or-latino,ACE1,cpDNA,
5,AU3586,AU3586201,2011-158.clean536p.fam,0,father,black-or-african-american,not-hispanic-or-latino,ACE1,cpDNA,
6,AU3586,AU3586202,2011-158.clean536p.fam,0,mother,black-or-african-american,not-hispanic-or-latino,ACE1,cpDNA,


In [21]:
table(aap$relationship)


         father  Father  mother  Mother sibling Sibling 
    417     105     179     107     195     304     638 

In [22]:
head(aap[aap$relationship == "",])

,FID,IID,file_name,aff,relationship,race,eth,grant,DNA.source,Notes
,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1529,A292_09,A292_09,2015-9017.clean170906.fam,NA,,,,Brain,Brain,
1530,AN00090,AN00090,2015-9017.clean170906.fam,NA,,,,Brain,Brain,
1531,AN00493,AN00493,2015-9017.clean170906.fam,NA,,,,Brain,Brain,
1532,AN00544,AN00544,2013-111A.ATN.clean170906.fam,NA,,,,Brain,Brain,
1533,AN00764,AN00764,2015-9017.clean170906.fam,NA,,,,Brain,Brain,
1534,AN01125,AN01125,2013-111A.ATN.clean170906.fam,NA,,,,Brain,Brain,


In [23]:
table(aap[aap$relationship == "", "grant"])


   Brain     CART Cryobank      HNP     Lord Pelphrey 
     119       96        6      106       82        8 

In [24]:
aap$relationship <- tolower(aap$relationship)

In [25]:
table(aap$relationship)


         father  mother sibling 
    417     284     302     942 

In [26]:
284+302+942

[1] 1528

In [27]:
aap_m <- aap[,c("FID", "IID", "relationship")]

In [28]:
colnames(aap_m) <- c("FID_aap", "IID", "relationship_aap")

In [29]:
head(dep_long)

FID,Relation,ID,relationship,IID
<chr>,<chr>,<chr>,<chr>,<chr>
AU0226,PROBAND_ID,AU0226302_AU0226302,sibling,AU0226302
AU0226,UNAFFECTED_ID,AU0226303_AU0226303,sibling,AU0226303
AU0253,PROBAND_ID,AU025304_AU025304,sibling,AU025304
AU0329,PROBAND_ID,AU032904_AU032904,sibling,AU032904
AU0347,PROBAND_ID,AU0347301_AU0347301,sibling,AU0347301
AU0347,PROBAND_ID,AU0347302_AU0347302,sibling,AU0347302


In [44]:
dep_m <- dep_long[,c("Relation", "ID", "relationship", "IID")]

In [31]:
dim(dep_m)

[1] 89387     4

In [46]:
dep_m <- unique(dep_m)
dim(dep_m)

[1] 81824     4

In [47]:
sum(aap_m$IID %in% dep_m$IID)

[1] 46

In [33]:
dfm <- left_join(dep_m, aap_m, by="IID")

In [34]:
head(dfm)

Relation,ID,relationship,IID,FID_aap,relationship_aap
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
PROBAND_ID,AU0226302_AU0226302,sibling,AU0226302,NA,NA
UNAFFECTED_ID,AU0226303_AU0226303,sibling,AU0226303,NA,NA
PROBAND_ID,AU025304_AU025304,sibling,AU025304,NA,NA
PROBAND_ID,AU032904_AU032904,sibling,AU032904,NA,NA
PROBAND_ID,AU0347301_AU0347301,sibling,AU0347301,NA,NA
PROBAND_ID,AU0347302_AU0347302,sibling,AU0347302,NA,NA


In [35]:
dim(dfm)

[1] 81828     6

In [36]:
dfm <- dfm[!is.na(dfm$relationship_aap),]

In [49]:
dim(dfm)

[1] 46  6

In [56]:
dpd <- fread("/u/project/geschwind/davidgib/ace/ptdt/ptdt_fam_structure_eur.txt")

In [57]:
head(dpd)

fID,probandID,fatherID,motherID,siblingID
<chr>,<chr>,<chr>,<chr>,<chr>
AU0226,AU0226302_AU0226302,NA,NA,AU0226303_AU0226303
AU0253,AU025304_AU025304,NA,NA,NA
AU0329,AU032904_AU032904,NA,NA,NA
AU0347,AU0347301_AU0347301,NA,NA,NA
AU0347,AU0347302_AU0347302,NA,NA,NA
AU0347,AU0347303_AU0347303,NA,NA,NA


In [58]:
# Use pivot_longer to reshape the data
dpd_long <- dpd %>%
  pivot_longer(cols = c(probandID, fatherID, motherID, siblingID), 
               names_to = "Relation", 
               values_to = "ID") %>%
  filter(!is.na(ID))  # Filter out rows where IDs are missing, if needed

In [59]:
head(dpd_long)

fID,Relation,ID
<chr>,<chr>,<chr>
AU0226,probandID,AU0226302_AU0226302
AU0226,siblingID,AU0226303_AU0226303
AU0253,probandID,AU025304_AU025304
AU0329,probandID,AU032904_AU032904
AU0347,probandID,AU0347301_AU0347301
AU0347,probandID,AU0347302_AU0347302


In [60]:
dpd_long <- dpd_long %>%
  mutate(relationship = case_when(
    Relation == "fatherID" ~ "father",
    Relation == "motherID" ~ "mother",
    Relation == "probandID" ~ "sibling",
    Relation == "siblingID" ~ "sibling"
  ))

In [62]:
head(dpd_long)

fID,Relation,ID,relationship
<chr>,<chr>,<chr>,<chr>
AU0226,probandID,AU0226302_AU0226302,sibling
AU0226,siblingID,AU0226303_AU0226303,sibling
AU0253,probandID,AU025304_AU025304,sibling
AU0329,probandID,AU032904_AU032904,sibling
AU0347,probandID,AU0347301_AU0347301,sibling
AU0347,probandID,AU0347302_AU0347302,sibling


In [63]:
colnames(dpd_long) <- c("FID", "Relation", "ID", "relationship")

In [64]:
dpd_long$IID <- sub(".*_", "", dpd_long$ID)

In [65]:
dpd_m <- dpd_long[,c("Relation", "ID", "relationship", "IID")]

In [67]:
dim(dpd_m)

[1] 1017    4

In [68]:
dpd_m <- unique(dpd_m)

In [69]:
dim(dpd_m)

[1] 728   4

In [70]:
dpm <- left_join(dpd_m, aap_m, by="IID")